In [ ]:
# Import FastMCP to create an MCP-compatible tool server
from fastmcp import FastMCP

# Import httpx for making asynchronous HTTP requests
import httpx

# Initialize the MCP server and give it a name "Weather"
mcp = FastMCP("Weather")

# Define an asynchronous tool that can be invoked by an MCP client or agent
@mcp.tool()
async def get_weather(location: str) -> str:
    """
    Get the weather for a given location using wttr.in API.
    
    Args:
        location (str): City name or location to fetch the weather for.
    
    Returns:
        str: A short weather description (e.g., "Karachi: 🌤 +38°C") or error message.
    """
    try:
        # Create the URL for wttr.in with simple 1-line format
        url = f"https://wttr.in/{location}?format=3"
        
        # Use async HTTP client to make the request
        async with httpx.AsyncClient() as client:
            response = await client.get(url)
            
            # Return weather text if successful
            if response.status_code == 200:
                return response.text.strip()
            else:
                return f"Could not fetch weather for {location}. Status: {response.status_code}"
    
    # Handle any unexpected exceptions and return the error message
    except Exception as e:
        return f"Error fetching weather: {str(e)}"
    
@mcp.tool()
def get_currency_value(currency_code: str) -> float:
    '''
    Return the current value of a currency against USD.
    :param currency_code: ISO currency code (e.g., "USD", "INR", "EUR")
    :return: current value of the currency against USD
    '''
    return {
        "USD": 1.0,     # US Dollar
        "INR": 0.012,   # Indian Rupee → 1 INR ≈ 0.012 USD
        "EUR": 1.09,    # Euro → 1 EUR ≈ 1.09 USD
    }.get(currency_code.upper(), 0.0)
# Start the MCP tool server on localhost:8000 using streamable HTTP transport
if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)

"""
client.py

This is the controller script that connects to multiple MCP tool servers
(weather and email), loads them into a LangGraph AI agent, and performs
automated tool calling using natural language instructions.

"""

# Import the MCP client adapter to connect with multiple MCP tool servers
from langchain_mcp_adapters.client import MultiServerMCPClient

# Import a pre-built LangGraph agent (ReAct logic) for tool reasoning
from langgraph.prebuilt import create_react_agent

# Import ChatGroq model wrapper for using Groq's LLMs (e.g., Qwen)
from langchain_groq import ChatGroq

# Used to load environment variables (e.g., GROQ_API_KEY)
from dotenv import load_dotenv
load_dotenv()  # Load .env file values into the environment

# Import asyncio to run the asynchronous workflow
import asyncio
from langchain.chat_models import init_chat_model
# Main asynchronous function that executes the AI agent logic
async def main():
    # Define the MCP tool servers and their configuration
    client = MultiServerMCPClient(
        {
            "weather": {
                "url": "http://localhost:8000/mcp",  # Weather tool server URL
                "transport": "streamable_http",      # Transport protocol
            },
           
        }
    )

    tools = await client.get_tools()

    import os
    os.environ["OPENAI_API_KEY"] = ""  # <-- paste your real key here

    # Initialize OpenAI GPT model
    model = init_chat_model(
    "gpt-3.5-turbo",
    model_provider="openai",
    temperature=0
    )
    # Create the LangGraph ReAct agent with tools and model
    agent = create_react_agent(model, tools)

    # Step 1: Ask the agent for current weather in Karachi
    weather_response = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": "what is the current weather in Hyderabad?"}
        ]
    })
    print("Weather response:", weather_response['messages'][-1].content)
    
    response1 = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": "Who is the pm of indian in one line?"}
        ]
    })
    print("Who is the pm of indian:", response1['messages'][-1].content)
    
    response2 = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": "What is the value of USD?"}
        ]
    })
    print("USD Value:", response2['messages'][-1].content)
    
    response3 = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": "What is the value of INR"}
        ]
    })
    print("INR Value:", response3['messages'][-1].content)


   

# Run the async workflow
asyncio.run(main())


Great 👍 I see your table (parent–agent RAG with multiple child agents: Weather App, Inventory, Order, Fulfillment, Store Info).

Right now your code only has **Weather** and a dummy **Currency tool**. To align with your architecture (from the picture), you need to extend the MCP server with multiple tools representing each **child agent use case**.

---

## 🔹 How to map the table → code

| From Picture (Child Agent)             | Tool Example                     |
| -------------------------------------- | -------------------------------- |
| Child Agent1 – Weather App             | ✅ Already done (`get_weather`)   |
| Child Agent2 – Inventory Microservice  | Tool: check product availability |
| Child Agent3 – Order Microservice      | Tool: place/cancel order         |
| Child Agent4 – Fulfillment Ops         | Tool: update shipment status     |
| Child Agent5 – Store Info Microservice | Tool: fetch store details        |

---

## 🔹 Extended MCP Server (`server.py`)

```python
# Import FastMCP
from fastmcp import FastMCP
import httpx

# Initialize MCP server
mcp = FastMCP("SupplyChainAgent")

# -----------------------------
# Child Agent1: Weather App
# -----------------------------
@mcp.tool()
async def get_weather(location: str) -> str:
    """Fetch short weather info from wttr.in"""
    try:
        url = f"https://wttr.in/{location}?format=3"
        async with httpx.AsyncClient() as client:
            resp = await client.get(url)
            return resp.text.strip() if resp.status_code == 200 else f"Error: {resp.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"

# -----------------------------
# Child Agent2: Inventory Microservice
# -----------------------------
@mcp.tool()
def check_inventory(product_id: str) -> str:
    """Check product availability (dummy)"""
    inventory = {"P001": 20, "P002": 0, "P003": 15}
    qty = inventory.get(product_id.upper(), None)
    if qty is None:
        return f"Product {product_id} not found"
    return f"Product {product_id} has {qty} units in stock"

# -----------------------------
# Child Agent3: Order Microservice
# -----------------------------
@mcp.tool()
def place_order(product_id: str, quantity: int) -> str:
    """Place an order (dummy microservice)"""
    return f"✅ Order placed for {quantity} units of {product_id}"

# -----------------------------
# Child Agent4: Fulfillment Ops
# -----------------------------
@mcp.tool()
def update_shipment(order_id: str, status: str) -> str:
    """Update shipment status (dummy)"""
    return f"🚚 Order {order_id} updated to status: {status}"

# -----------------------------
# Child Agent5: Store Info Microservice
# -----------------------------
@mcp.tool()
def get_store_info(store_id: str) -> str:
    """Fetch store details (dummy microservice)"""
    stores = {
        "S001": "Berhampur Store, Odisha",
        "S002": "Hyderabad Store, Telangana"
    }
    return stores.get(store_id.upper(), f"Store {store_id} not found")

# -----------------------------
# Run MCP Server
# -----------------------------
if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)
```

---

## 🔹 Update Client (`client.py`)

Your `client.py` already connects with `MultiServerMCPClient` and loads all tools.
Now you can **ask multi-domain questions** like:

```python
# Ask weather
resp1 = await agent.ainvoke({
    "messages": [{"role": "user", "content": "What is the weather in Berhampur?"}]
})
print("Weather:", resp1["messages"][-1].content)

# Check inventory
resp2 = await agent.ainvoke({
    "messages": [{"role": "user", "content": "Check inventory for P001"}]
})
print("Inventory:", resp2["messages"][-1].content)

# Place order
resp3 = await agent.ainvoke({
    "messages": [{"role": "user", "content": "Place order for 2 units of P003"}]
})
print("Order:", resp3["messages"][-1].content)

# Update shipment
resp4 = await agent.ainvoke({
    "messages": [{"role": "user", "content": "Update order O100 to Delivered"}]
})
print("Shipment:", resp4["messages"][-1].content)

# Get store info
resp5 = await agent.ainvoke({
    "messages": [{"role": "user", "content": "Where is store S002 located?"}]
})
print("Store Info:", resp5["messages"][-1].content)
```

---

⚡ Now your setup mirrors the **Parent → Child agent design** from the picture:

* Parent = LangGraph ReAct Agent (RAG, memory, reasoning)
* Children = MCP tools (Weather, Inventory, Orders, Fulfillment, Store Info)

---

👉 Do you want me to **merge the Weather + Currency code you already had** with this full multi-agent setup, so you can run everything in one MCP server?
